# Création des fichiers de noeuds et d'arrêtes pour reproduire nos deux graphes sur gephi

In [ ]:
import pandas as pd
import ast


In [ ]:
# Jeu de données restreint au citations internes
df = pd.read_csv("openalex_vietnam_new_year_family.csv")
df

In [10]:
df["referenced_ids_internal"] # mauvais format hélas

0       ['https://openalex.org/W2048335617', 'https://...
1                    ['https://openalex.org/W1988996614']
2       ['https://openalex.org/W2890659721', 'https://...
3       ['https://openalex.org/W2133466515', 'https://...
4       ['https://openalex.org/W2132908530', 'https://...
                              ...                        
2460                 ['https://openalex.org/W1602209609']
2461                 ['https://openalex.org/W1978007171']
2462                 ['https://openalex.org/W2073255868']
2463                 ['https://openalex.org/W2053341678']
2464    ['https://openalex.org/W845819200', 'https://o...
Name: referenced_ids_internal, Length: 2465, dtype: object

In [11]:
# transformer le type de la colonne refrenced_ids_internal en list
def parse_list(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return x

df["referenced_ids_internal"] = df["referenced_ids_internal"].apply(parse_list)


## 1. Graphe des citations

In [ ]:
# source : l'article (id) ; targets : les articles (referenced_ids_internal)
# comme il y possiblement plusieurs citations : explode puis
edges = (df.explode("referenced_ids_internal")[["id", "referenced_ids_internal"]].rename(columns={"id": "source", "referenced_ids_internal": "target"}).dropna().drop_duplicates())
nodes = df[["id","lang_type","publication_year"]].drop_duplicates()

In [ ]:
# ajout du in-degree = nombre de citations reçues
in_deg = edges.groupby("target").size().rename("in_degree")
nodes = nodes.set_index("id")

nodes = nodes.join(in_deg, how="left")

nodes["in_degree"] = nodes["in_degree"].fillna(0).astype(int)

nodes = nodes.reset_index()

In [ ]:
# pour éviter de se retrouver avec des noeuds en plus 
valid_ids = set(nodes["id"])
edges = edges[ edges["source"].isin(valid_ids) & edges["target"].isin(valid_ids)]



In [ ]:
edges.to_csv("edges_citation.csv", index=False)
nodes.to_csv("nodes_citation.csv", index=False)

## 2. Graphe bipartite des articles et de leur topic principal

In [ ]:
df2 = df.drop(columns=['publication_year', 'language', 'type',
       'institution_countries', 'institutions', 'domains', 'fields',
       'subfields', 'topics',  'primary_topic_field_name',
       'primary_topic_subfield_name', 'primary_topic_domain_name',
       'referenced_ids', 'referenced_count', 'referenced_ids_internal'])

In [ ]:

# arrêtes bipartites
edges2 = pd.DataFrame({"source": df2["id"],"target": "topic:" + df2["primary_topic_name"].astype(str)})
edges2.to_csv("edges_bipartite_topic2.csv", index=False)


# noeuds publications
nodes_pub = pd.DataFrame({"id": df2["id"],"label": "","type": "publication","lang_type": df2["lang_type"]})

# noeuds topics principaux
nodes_topics = pd.DataFrame({"id": "topic:" + df2["primary_topic_name"].astype(str),"label": df2["primary_topic_name"],"type": "topic","lang_type": "topic"  }).drop_duplicates()

# Combinaison des noeuds et colonne indicatrice
nodes2 = pd.concat([nodes_pub, nodes_topics], ignore_index=True)
nodes2["is_topic"] = nodes2["type"] == "topic"


compromis d'affichage lisible : afficher les labels des 50 topics principaux les plus présents seulement

In [ ]:
top_topics = list(df2["primary_topic_name"].value_counts()[:50].index)

In [ ]:
# horreurs qui marchent
label2 = []
for i in nodes2["label"] : 
    print(i)
    if i in top_topics : 
        label2.append(i)
    else :
        label2.append("")

nodes2["label"] = nodes2["label"].replace("", "rien")
nodes2["labels2"] = label2
nodes2 = nodes2.drop(columns= ["label"])
nodes2 = nodes2.rename(columns={"labels2" : "label"})

In [ ]:
nodes2.to_csv("nodes_bipartite_topic2.csv", index=False)